# Biomedical AI Teaching Lab

从 U-Net 的一次 CPU 实验开始，检查数据、参数与日志。课程条目覆盖 137 个模型，但并非都已具备本地适配器；原代码、论文移植、机制演示、原版路径分别标记。两步或二十步训练不能证明原论文性能。

[教学网站](https://happymie.github.io/teaching/) · [覆盖与限制](https://github.com/happymie/happymie.github.io/blob/main/teaching/docs/COVERAGE.md)

在 Colab 中选择普通 CPU runtime 即可。此 notebook 会在工作目录克隆公开仓库并安装固定 CPU 依赖；不获取任何门控权重或患者数据。

In [ ]:
import os, sys, subprocess
from pathlib import Path
repo = Path.cwd() / "happymie.github.io"
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/happymie/happymie.github.io.git", str(repo)], check=True)
lab = repo / "teaching"
assert lab.is_dir(), "Existing destination is not this teaching repository"
os.chdir(lab)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-cpu.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."], check=True)


## 1. 先看实现状态与数据

CLI 使用新解释器执行，避免 notebook 内已经导入的库与新安装版本混在一起。第 006 项是缩小 U-Net 的结构实现；配置/差异由实际注册表提供。

In [ ]:
subprocess.run([sys.executable, "-m", "biomed_lab", "info", "006"], check=True)
subprocess.run([sys.executable, "-m", "biomed_lab", "data", "006", "--inspect"], check=True)


## 2. 运行 20 步，并保留资源记录

默认两线程、批次 2。内存与时间检查是训练/验证边界上的软预算；该小合成数据实验不代表真实临床训练。

In [ ]:
subprocess.run([sys.executable, "-m", "biomed_lab", "run", "006", "--steps", "20", "--threads", "2", "--batch-size", "2", "--device", "cpu"], check=True)


## 3. 读实验 SQLite

训练和独立合成验证分别保存。测试划分在此保留。参数量、损失变化与内存各回答不同问题。

In [ ]:
import json, sqlite3
result_path = max(Path("runs").glob("*/result.json"), key=lambda p: p.stat().st_mtime)
result = json.loads(result_path.read_text())
print({k: result[k] for k in ["title", "status", "parameters", "steps_completed", "observed_peak_rss_mb", "memory_measurement"]})
with sqlite3.connect(result_path.parent / "experiment.sqlite3") as db:
    rows = db.execute("SELECT step, split, loss, elapsed_seconds, rss_mb FROM metrics ORDER BY step, split").fetchall()
for row in rows:
    print(row)


## 4. 只改变一个参数

把 U-Net 的基础宽度减为 4，保留其他条件。比较参数量和当前机器中的内存记录；二十步上的损失差异不足以判断最终性能。

In [ ]:
subprocess.run([sys.executable, "-m", "biomed_lab", "run", "006", "--steps", "20", "--set", "width=4", "--threads", "2"], check=True)


## 5. 选下一课

运行 `list` 后，先选择已实现条目。对于 `source_route`，先读原作者路径及缺口；没有适配器时执行命令会明确报错，不会替换模型。

每次学习提交：计算图、默认配置记录、一次有解释的改变、以及它尚不能证明的结论。

In [ ]:
subprocess.run([sys.executable, "-m", "biomed_lab", "list"], check=True)
